Steps to preprocess MODIS data for phenology tracking. Note: GEE runs into memory issues when you try manipulating any of the big data for plotting/viewing/etc. So I am exporting these as geotiffs and reading them in as rasters to python to be able to look at the processing steps.

Author: Tayler LaSharr

Date: September 2024

In [1]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import os
import pprint

import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

In [2]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '11-01'
end_month_day = '03-30'

## threshold for snow in winter to get the 'baseline' value in winter
snow_threshold = 50

## get the time period we care about
start_date_data = '2015-01-01'
end_date_data = '2023-12-31'


Read in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons)

In [3]:
## load in a shapefile or area of interest
## this is an assest that is already loaded into GEE

# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## dry af
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()

## load the MODIS NDVI data (before referencing its projection)
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA')

## function to calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add an NDVI band
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()

reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounfinh box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## daily snow cover
modis_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDSI_Snow_Cover') \
                .map(lambda image: image.clip(reprojected_bounding_box))

coordidor_projection = md_cooridor.first().geometry().projection()

## make sure the time property is there
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )

modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))

## just check on a couple of things
print('Projection of MODIS NDVI data:', modis_projection.getInfo())
print('Reprojected Bounding Box:', reprojected_bounding_box.getInfo())
print("Bands available in the first image:", modis_ndvi.first().bandNames().getInfo())


Projection of MODIS NDVI data: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Reprojected Bounding Box: {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-10023056.371103, 4100000.2730249995], [-9953959.851255, 4100000.2730249995], [-9904838.987216, 4141734.4837789997], [-9973594.529121, 4141734.4837789997], [-10023056.371103, 4100000.2730249995]]]}
Bands available in the first image: ['sur_refl_b01', 'sur_refl_b02', 'QA', 'NDVI']


In [4]:
## this step exports a bounding box for your study area. This will eventually be unnecessary, I was using it
## to crop the rasters to to get rid of weird values that were happening outside of the study area
## so I could look at the trends through time. Can comment this out if unneeded.

bounding_box_fc = ee.FeatureCollection(ee.Feature(bounding_box))

task = ee.batch.Export.table.toDrive(
    collection=bounding_box_fc,
    description='BoundingBoxExport',
    folder='DroughtSurfing',  ## Google Drive folder where you want to save the shapefile
    fileNamePrefix='bounding_box',  ## shapefile name
    fileFormat='SHP'  ## format for exporting
)

## start the export task
task.start()

print('Bounding box export to Google Drive')

Bounding box export to Google Drive


Step 1. Remove low-quality observations with the QA band.

In [5]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow

def quality_mask_with_optional_snow(image, allow_snow=True):

    ## get the QA band
    quality = image.select('QA')

    ## Bits 0-1: MODLAND QA bits (0 = good, 1 = less than ideal, 2 = cloud effects, 3 = other)
    modland_qa = quality.bitwiseAnd(3).lte(1)  # Keep pixels with quality 0 or 1

    ## Bits 2-5: Band 1 data quality (focus on NDVI bands)
    band1_quality = quality.rightShift(2).bitwiseAnd(15).lte(8)  
    
    ## Bits 6-9: Band 2 data quality (focus on NDVI bands)
    band2_quality = quality.rightShift(6).bitwiseAnd(15).lte(8)  
    
    ## Bit 14: Snow/Ice (0 = no snow/ice, 1 = snow/ice)
    snow_ice = quality.rightShift(14).bitwiseAnd(1).eq(1)  
    
    ## create a mask combining MODLAND QA and Band 1/2 quality
    mask = modland_qa.And(band1_quality).And(band2_quality)
    ndvi_masked = image.updateMask(mask).select(['NDVI', 'QA'])

    ## deal with snow (if allowed)
    if not allow_snow:
        ## if snow is not allowed, mask snow/ice pixels
        ndvi_masked = ndvi_masked.updateMask(snow_ice.Not())

    ## remove NDVI outliers (NDVI values should be between -1 and 1)
    ndvi_outlier_masked = ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(1)))

    return ndvi_outlier_masked


## apply the mask to the modis NDVI collection
modis_ndvi_filtered = modis_ndvi.map(lambda img: quality_mask_with_optional_snow(img, allow_snow=True))

## check the number of images (make sure it's running things and not getting rid of everything)
print("Number of images in the filtered collection:", modis_ndvi_filtered.size().getInfo())


Number of images in the filtered collection: 414


In [6]:
## this checks for snow in the previous 3 images to see if the NDVI data are real or not. This is methodology
## that Merkle develped to update the Bischof methodology. It helps deal with NDVI values that were being estimated
## in areas where snow still was covering the ground.

## this may be something that we want to take out for some of the desert systems. I am currently running it
## in the processing steps to see if it will cause problems, so that we can use the same steps for processing
## all data if possible.

def mask_snow_free_pixels(ndvi_collection):

    ## convert the NDVI collection to a list of images
    ndvi_list = ndvi_collection.toList(ndvi_collection.size())
    
    def apply_mask_to_ndvi(index):
        # Get the current NDVI image
        ndvi_image = ee.Image(ndvi_list.get(index))

        # Get the QA band from the current NDVI image
        quality = ndvi_image.select('QA')

        ## bit 14: Snow/Ice (0 = no snow/ice, 1 = snow/ice) for the current image
        snow_current = quality.rightShift(14).bitwiseAnd(1).eq(0)  # 0 = snow-free

        ## check if there are previous images in the list (i.e., ensure index is valid)
        ## get the snow condition for the previous image (index - 1)
        previous_1 = ee.Algorithms.If(ee.Number(index).gte(1), ndvi_list.get(ee.Number(index).subtract(1)), None)
        snow_previous_1 = ee.Image(ee.Algorithms.If(previous_1, ee.Image(previous_1).select('QA').rightShift(14).bitwiseAnd(1).eq(0), ee.Image(1)))

        ## get the snow condition for the image before the previous one (index - 2)
        previous_2 = ee.Algorithms.If(ee.Number(index).gte(2), ndvi_list.get(ee.Number(index).subtract(2)), None)
        snow_previous_2 = ee.Image(ee.Algorithms.If(previous_2, ee.Image(previous_2).select('QA').rightShift(14).bitwiseAnd(1).eq(0), ee.Image(1)))

        ## create a snow-free mask: A pixel is snow-free if it's snow-free in all three time steps (current, previous_1, previous_2)
        snow_free_mask = snow_current.And(snow_previous_1).And(snow_previous_2)

        ## apply the snow-free mask to the NDVI image
        return ndvi_image.updateMask(snow_free_mask)

    ## map over the image indices and apply the mask
    indices = ee.List.sequence(0, ndvi_collection.size().subtract(1))
    return ee.ImageCollection(indices.map(apply_mask_to_ndvi))

## run the function to get the snow-free NDVI collection
modis_ndvi_filtered = mask_snow_free_pixels(modis_ndvi_filtered)


Step 2. Set winter baselines dynamically since winter conditions are variable year to year. This differs from the Bischof and Merkle methods because winter is not consistent and some winters have snow while others do not. For this step we basically go through several different substeps. First, we define the winter period and add that attribute to the image collection. Then we calculate the 'baseline' winter values during that period for all cells that have less than a set amount of snow cover. Then we fill all empty cells during the winter period with the baseline value. Then, if there is snow covering the cells in winter, we will replace them with the winter baseline. Finally, to further smooth things in winter, if the image before and after the current image is set to the winter baseline, we will also set the current image to the baseline. This will prevent weird bumps from popping up throuhgout winter.

We don't set the winter values to the "baseline" for the entire winter because in desert systems there might be green that is more variable.

In [7]:
## get the baseline value to replace Null values in winter with
## need to make sure that this is doing what I want, something was happening where some of the 'baseline' values were -2000, so I added an
## argument that would not allow negative values to be considered when calculating the 0.025 quantile, but need to confirm that is not
## messing up how the baseline is calculated. check with Ellen on her thoughts here.

def calculate_baseline(image_collection, modis_snow, snow_threshold, quantile=0.025):
   
    def mask_snow_and_negatives(ndvi_image):
        
        ## get the date of the NDVI image
        image_date = ee.Date(ndvi_image.get('system:time_start'))

        ## create a buffer period around the NDVI image date (e.g., 16 days)
        buffer_period = 16
        snow_images = modis_snow.filterDate(image_date.advance(-buffer_period, 'day'), image_date.advance(buffer_period, 'day')).mean()

        ## create a snow mask where NDSI_Snow_Cover >= snow_threshold
        snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold)

        ## apply the snow mask to the NDVI image (exclude values where snow cover exceeds the threshold)
        return ndvi_image.updateMask(snow_mask.Not()).updateMask(ndvi_image.gte(0))

    ## apply the snow mask to each image in the collection
    snow_and_negatives_masked = image_collection.map(mask_snow_and_negatives)

    ## calculate the number of images used to get the baseline
    image_count = snow_and_negatives_masked.select('NDVI').reduce(ee.Reducer.count()).rename('image_count')

    ## calculate the 0.025 quantile (or any specified quantile) for NDVI, excluding snow-covered pixels
    baseline_ndvi = snow_and_negatives_masked.select('NDVI').reduce(ee.Reducer.percentile([quantile * 100])).rename('NDVI_baseline')

    return baseline_ndvi.addBands(image_count)

## calculate the baseline for the entire collection
winter_baseline_ndvi = calculate_baseline(modis_ndvi_filtered, modis_snow, snow_threshold=snow_threshold)
print("Winter Baseline NDVI calculated.")


Winter Baseline NDVI calculated.


In [8]:
## now generate a snow mask function, this function will create a mask if the snow cover on that pixel is equal to or greater than the threshold we set
def generate_snow_mask(ndvi_image, modis_snow, snow_threshold):

    ## get the date of the NDVI image
    image_date = ee.Date(ndvi_image.get('system:time_start'))

    ## define a buffer period to search for snow images
    buffer_period = 16  # e.g., 16 days buffer

    ## pull out the images that are within the buffer
    snow_images = modis_snow.filterDate(image_date.advance(-buffer_period, 'day'), image_date.advance(buffer_period, 'day')).mean()

    ## if the snow cover layer is greater than or equal to the threshold, create a snow mask
    snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold)
    return snow_mask

## now, during winter, set all empty cells and cells with values less than the baseline to the baseline
def fill_and_correct_winter_values(ndvi_image, baseline_image, modis_snow, snow_threshold, start_month_day, end_month_day):

    ## get the date of the year
    image_date = ee.Date(ndvi_image.get('system:time_start'))
    year = image_date.get('year')

    ## get the start and end dates for the winter period
    start_date = ee.Date.fromYMD(year, ee.Number.parse(start_month_day.split('-')[0]), ee.Number.parse(start_month_day.split('-')[1]))
    end_date = ee.Date.fromYMD(year.add(1), ee.Number.parse(end_month_day.split('-')[0]), ee.Number.parse(end_month_day.split('-')[1]))

    ## figure out if it's winter or not
    is_winter = image_date.millis().gte(start_date.millis()).And(image_date.millis().lte(end_date.millis()))

    ## apply the correction if necessary (if it's less than the baseline or empty)
    def apply_corrections(image):
        ndvi_band = image.select('NDVI')

        ## apply snow mask corrections year-round
        snow_mask = generate_snow_mask(image, modis_snow, snow_threshold)

        ## then, during winter, also correct for missing and below-baseline values
        corrected_ndvi = ndvi_band.where(snow_mask, baseline_image.select('NDVI_baseline')) \
                                  .where(ndvi_band.lt(baseline_image.select('NDVI_baseline')), baseline_image.select('NDVI_baseline'))
        return image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)

    ## apply corrections if it's winter; outside winter, apply a less aggressive snow mask correction
    return ee.Image(ee.Algorithms.If(is_winter, apply_corrections(ndvi_image), ndvi_image))


## apply the function to correct and fill NDVI during winter, with snow mask corrections applied year-round
modis_ndvi_corrected = modis_ndvi_filtered.map(
    lambda img: fill_and_correct_winter_values(img, winter_baseline_ndvi, modis_snow, snow_threshold, start_month_day, end_month_day)
)


In [9]:
def apply_neighbor_baseline_correction(ndvi_image, baseline_image, all_images, target_crs, target_scale):
    
    ## reproject the current NDVI image to the target CRS and scale
    ndvi_image = ndvi_image.reproject(crs=target_crs, scale=target_scale)

    ## reproject all images to the target CRS and scale
    all_images_list = all_images.toList(all_images.size())
    image_index = all_images_list.indexOf(ndvi_image)

    previous_image = ee.Image(ee.Algorithms.If(image_index.gt(0), all_images_list.get(image_index.subtract(1)), None))
    next_image = ee.Image(ee.Algorithms.If(image_index.lt(all_images.size().subtract(1)), all_images_list.get(image_index.add(1)), None))

    ## only reproject if the image is not None
    previous_image = ee.Algorithms.If(previous_image, ee.Image(previous_image).reproject(crs=target_crs, scale=target_scale), None)
    next_image = ee.Algorithms.If(next_image, ee.Image(next_image).reproject(crs=target_crs, scale=target_scale), None)

    def is_baseline(image):
        return image.select('NDVI').eq(baseline_image.select('NDVI_baseline')).reduceRegion(
            reducer=ee.Reducer.allNonZero(),
            geometry=image.geometry(),
            scale=target_scale
        ).values().get(0)

    prev_set_to_baseline = ee.Algorithms.If(previous_image, is_baseline(ee.Image(previous_image)), ee.Number(0))
    next_set_to_baseline = ee.Algorithms.If(next_image, is_baseline(ee.Image(next_image)), ee.Number(0))

    ## determine if both neighboring images are set to the baseline
    both_neighbors_set_to_baseline = ee.Number(prev_set_to_baseline).And(ee.Number(next_set_to_baseline))

    ## define corrections based on neighbors
    def correct_to_baseline():
        return baseline_image.select('NDVI_baseline').rename('NDVI')

    def keep_current_ndvi():
        return ndvi_image.select('NDVI')

    ## apply corrections based on neighbors
    corrected_ndvi = ee.Image(ee.Algorithms.If(both_neighbors_set_to_baseline, correct_to_baseline(), keep_current_ndvi()))
    return ndvi_image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)


## apply_neighbor_check_to_collection function to run it for the entire image collection
def apply_neighbor_check_to_collection(image_collection, baseline_image, target_crs, target_scale):
    all_images = image_collection.sort('system:time_start')
    
    def check_neighbors(image):
        corrected_image = apply_neighbor_baseline_correction(image, baseline_image, all_images, target_crs, target_scale)
        return corrected_image.select(['NDVI']).rename('NDVI')
    corrected_collection = image_collection.map(check_neighbors)
    return corrected_collection


## set the target projection and scale from your baseline NDVI or the first image in your collection
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

## apply the neighbor correction with consistent projections
modis_ndvi_corrected = apply_neighbor_check_to_collection(modis_ndvi_corrected, winter_baseline_ndvi, target_crs, target_scale)


Step 3. Apply a smoothing function (moving median window with a search window of 3). 

IMPORTANT: Jerod's steps include running the moving mdeian fitler BEFORE scaling and flooring the data to the baseline. I need to check in with him about this and why he did it that way instead of after.

In [10]:
def apply_image_median(image_collection, window_size=3):
    
    image_collection = image_collection.map(lambda img: img.select('NDVI').toShort())

    ## define a function to apply median filtering using a fixed number of images around each image
    def smooth_image(current_image):
        
        ## get the list of all images in the collection, sorted by date
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())

        ## get the index of the current image
        image_index = all_images_list.indexOf(current_image)
        
        ## calculate half of the window size
        half_window = ee.Number(window_size // 2)

        ## define the start and end indices for the window (centered on the current image)
        ## ensure start_index is not less than 0
        start_index = ee.Number(image_index).subtract(half_window).max(0)
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## create a list of images within the window
        window_images = all_images_list.slice(start_index, end_index.add(1))
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median of the selected images
        median_image = window_collection.median()
        median_image = median_image.toShort()

        ## return the median-smoothed image, keeping the original properties
        return median_image.copyProperties(current_image, current_image.propertyNames())

    ## apply the smoothing function over the entire image collection
    smoothed_collection = image_collection.map(smooth_image)
    return smoothed_collection

## apply the smoothing function to the processed NDVI collection
smoothed_ndvi = apply_image_median(modis_ndvi_corrected, window_size=3)
smoothed_ndvi = smoothed_ndvi.map(lambda img: img.reproject(crs=target_crs, scale=target_scale))



Export the data (at each time step so we can plot and look at them)

In [11]:

## the raw data and the filtered data can be exported as a single stack
def reproject_and_export(image_collection, file_name, folder_name, target_crs, scale):
    try:

        ## reproject each image in the collection to the target CRS
        def reproject_image(image):
            return image.reproject(crs=target_crs, scale=scale)

        ## apply reprojection to the entire image collection
        reprojected_collection = image_collection.map(reproject_image)

        ## stack all reprojected images as bands in a single multi-band image
        stacked_image = reprojected_collection.toBands()

        ## get the projection of the reprojected images (just for logging purposes)
        reprojected_projection = reprojected_collection.first().projection()

        ## export the stacked image with the reprojected CRS and scale
        task = ee.batch.Export.image.toDrive(
            image=stacked_image,
            description=f'{file_name}_stack',
            folder=folder_name,  # Specify the folder in your Google Drive
            fileNamePrefix=file_name,  # Name for the exported file
            region=bounding_box,  # You can use the bounding box or full extent (omit for full extent)
            scale=scale,  # Set the scale to a reasonable value (e.g., 30 meters for WGS84)
            maxPixels=1e13,  # Avoid pixel limit issues by setting a high value
            fileFormat='GeoTIFF'  # Export as GeoTIFF
        )
        task.start()
        print(f'Exporting {file_name}_stack to Google Drive in {target_crs} with scale {scale}.')

    except Exception as e:
        print(f'Error exporting {file_name}: {e}')

## specify the target CRS and scale
target_crs = 'EPSG:4326'  # WGS84
scale = 250 # (250 is the MODIS scale)

## export using WGS84 as the target CRS
reproject_and_export(modis_ndvi.select('NDVI'), 'modis_ndvi', 'DroughtSurfing', target_crs, scale)
reproject_and_export(modis_ndvi_filtered, 'modis_ndvi_filtered', 'DroughtSurfing', target_crs, scale)

## takes about ~30 minutes to export the images for the Sublette herd area (but will fail in ~2 minutes if it is not working).
## Open up GEE (https://code.earthengine.google.com/) to see if it's working and it will give you errors if it doesn't work.


Exporting modis_ndvi_stack to Google Drive in EPSG:4326 with scale 250.
Exporting modis_ndvi_filtered_stack to Google Drive in EPSG:4326 with scale 250.


In [12]:
def export_collection_by_year(image_collection, file_name_prefix, start_year, end_year, folder_name, target_crs, scale):
    try:

        ## loop through each year and filter the collection by year
        for year in range(start_year, end_year + 1):
            
            ## filter the collection for the current year
            yearly_collection = image_collection.filterDate(f'{year}-01-01', f'{year}-12-31')

            ## check if the collection for the year is empty
            if yearly_collection.size().getInfo() > 0:

                ## reproject each image in the collection to the target CRS
                def reproject_image(image):
                    reprojected = image.reproject(crs=target_crs, scale=scale)
                    return reprojected.toInt16()  # Cast to Int16 after reprojection

                ## apply reprojection to the entire yearly collection
                reprojected_yearly_collection = yearly_collection.map(reproject_image)

                ## stack all reprojected images as bands in a single multi-band image
                stacked_image = reprojected_yearly_collection.toBands()

                ## export the stacked image
                task = ee.batch.Export.image.toDrive(
                    image=stacked_image,
                    description=f'{file_name_prefix}_{year}_stack',
                    folder=folder_name,  # Specify the folder in your Google Drive
                    fileNamePrefix=f'{file_name_prefix}_{year}',  # Name for the exported file
                    region=bounding_box,  # Original bounding box or the full extent
                    scale=scale,  # Set the scale for the reprojected CRS (e.g., 30 meters for WGS84)
                    maxPixels=1e13,  # Set to a high value to avoid pixel limit issues
                    fileFormat='GeoTIFF'  # Export as GeoTIFF
                )

                task.start()
                print(f'Exporting {file_name_prefix}_{year}_stack to Google Drive.')
            else:
                print(f"No images found for {year} in {file_name_prefix}.")

    except Exception as e:
        print(f'Error exporting {file_name_prefix}: {e}')

## specify the target CRS and scale
target_crs = 'EPSG:4326'  # WGS84
scale = 250 # (250 is the MODIS scale)

# Export the corrected and smoothed NDVI data by year
export_collection_by_year(modis_ndvi_corrected.select('NDVI'), 'modis_ndvi_corrected', 2015, 2023, 'DroughtSurfing', target_crs, scale)
export_collection_by_year(smoothed_ndvi, 'smoothed_ndvi', 2015, 2023, 'DroughtSurfing', target_crs, scale)


Exporting modis_ndvi_corrected_2015_stack to Google Drive.
Exporting modis_ndvi_corrected_2016_stack to Google Drive.
Exporting modis_ndvi_corrected_2017_stack to Google Drive.
Exporting modis_ndvi_corrected_2018_stack to Google Drive.
Exporting modis_ndvi_corrected_2019_stack to Google Drive.
Exporting modis_ndvi_corrected_2020_stack to Google Drive.
Exporting modis_ndvi_corrected_2021_stack to Google Drive.
Exporting modis_ndvi_corrected_2022_stack to Google Drive.
Exporting modis_ndvi_corrected_2023_stack to Google Drive.
Exporting smoothed_ndvi_2015_stack to Google Drive.
Exporting smoothed_ndvi_2016_stack to Google Drive.
Exporting smoothed_ndvi_2017_stack to Google Drive.
Exporting smoothed_ndvi_2018_stack to Google Drive.
Exporting smoothed_ndvi_2019_stack to Google Drive.
Exporting smoothed_ndvi_2020_stack to Google Drive.
Exporting smoothed_ndvi_2021_stack to Google Drive.
Exporting smoothed_ndvi_2022_stack to Google Drive.
Exporting smoothed_ndvi_2023_stack to Google Drive.
